In [1]:
import os
import numpy as np
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# =========================
# CONFIG
# =========================
DATASET_ROOT = "/home/feliciano/Downloads/Preliminary Data/AllFish_2secSplit"
SR = 22050
DURATION = 2.0
TARGET_LEN = int(SR * DURATION)
N_MELS = 64
BATCH_SIZE = 32
EPOCHS = 20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_STATE = 42

# =========================
# AUDIO UTILS
# =========================
def load_fixed_length(wav_path):
    y, _ = librosa.load(wav_path, sr=SR)

    if len(y) < TARGET_LEN:
        y = np.pad(y, (0, TARGET_LEN - len(y)))
    else:
        y = y[:TARGET_LEN]

    return y

# =========================
# DATASET
# =========================
class FishDataset(Dataset):
    def __init__(self, files, labels):
        self.files = files
        self.labels = labels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        y = load_fixed_length(self.files[idx])

        mel = librosa.feature.melspectrogram(
            y=y,
            sr=SR,
            n_mels=N_MELS,
            hop_length=512
        )
        mel = librosa.power_to_db(mel, ref=np.max)

        mel = torch.tensor(mel, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return mel, label

# =========================
# LOAD FILES
# =========================
files, labels = [], []

for label in sorted(os.listdir(DATASET_ROOT)):
    class_dir = os.path.join(DATASET_ROOT, label)
    if not os.path.isdir(class_dir):
        continue

    for f in os.listdir(class_dir):
        if f.endswith(".wav"):
            files.append(os.path.join(class_dir, f))
            labels.append(label)

le = LabelEncoder()
labels = le.fit_transform(labels)

# =========================
# SPLIT 70 / 20 / 10
# =========================
f_train, f_tmp, y_train, y_tmp = train_test_split(
    files, labels, test_size=0.30, stratify=labels, random_state=RANDOM_STATE
)

f_test, f_val, y_test, y_val = train_test_split(
    f_tmp, y_tmp, test_size=1/3, stratify=y_tmp, random_state=RANDOM_STATE
)

train_dl = DataLoader(FishDataset(f_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(FishDataset(f_val, y_val), batch_size=BATCH_SIZE)
test_dl  = DataLoader(FishDataset(f_test, y_test), batch_size=BATCH_SIZE)

# =========================
# MODEL
# =========================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.net(x)
        return self.fc(x.view(x.size(0), -1))

model = CNN(len(le.classes_)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# =========================
# TRAIN
# =========================
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss / len(train_dl):.4f}")

# =========================
# TEST
# =========================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for x, y in test_dl:
        preds = model(x.to(DEVICE)).argmax(1).cpu().numpy()
        y_true.extend(y.numpy())
        y_pred.extend(preds)

print("\n=== DL RESULTS (TEST SET) ===")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average="macro"))
print("Recall   :", recall_score(y_true, y_pred, average="macro"))
print("F1       :", f1_score(y_true, y_pred, average="macro"))


/home/feliciano/anaconda3/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Epoch 1/20 | Loss: 0.0134
Epoch 2/20 | Loss: 0.0008
Epoch 3/20 | Loss: 0.0004
Epoch 4/20 | Loss: 0.0005
Epoch 5/20 | Loss: 0.0002
Epoch 6/20 | Loss: 0.0002
Epoch 7/20 | Loss: 0.0001
Epoch 8/20 | Loss: 0.0002
Epoch 9/20 | Loss: 0.0001
Epoch 10/20 | Loss: 0.0002
Epoch 11/20 | Loss: 0.0002
Epoch 12/20 | Loss: 0.0001
Epoch 13/20 | Loss: 0.0001
Epoch 14/20 | Loss: 0.0000
Epoch 15/20 | Loss: 0.0001
Epoch 16/20 | Loss: 0.0001
Epoch 17/20 | Loss: 0.0001
Epoch 18/20 | Loss: 0.0001
Epoch 19/20 | Loss: 0.0001
Epoch 20/20 | Loss: 0.0000

=== DL RESULTS (TEST SET) ===
Accuracy : 0.9999211895129512
Precision: 0.9999707773232028
Recall   : 0.9998995042208227
F1       : 0.9999351298165661
